In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
import sys
sys.path.append("..") 
from processing import  load_UCI_dataset, print_metrics, plot_regression_results, evaluate_minirocket_vitaldb

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(
    WINDOW_SIZE=625,
    STEP_SIZE=312
)

BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints"

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
from sktime.transformations.panel.rocket import MiniRocket

rocket = MiniRocket(num_kernels=10000, random_state=42)

rocket.fit(X_train)

In [ ]:
X_train_rocket = rocket.transform(X_train)
X_val_rocket = rocket.transform(X_val)
X_test_rocket = rocket.transform(X_test)

print("X_train_rocket:", X_train_rocket.shape)
print("X_val_rocket:", X_val_rocket.shape)
print("X_test_rocket:", X_test_rocket.shape)

In [ ]:
np.save("X_train_minirocket.npy", X_train_rocket)
np.save("X_val_minirocket.npy", X_val_rocket)
np.save("X_test_minirocket.npy", X_test_rocket)

In [ ]:
import numpy as np

X_train_rocket = np.load("X_train_minirocket.npy")
X_val_rocket = np.load("X_val_minirocket.npy")
X_test_rocket = np.load("X_test_minirocket.npy")

In [ ]:
y_train_sbp = y_train[:, 0]
y_train_dbp = y_train[:, 1]

y_val_sbp = y_val[:, 0]
y_val_dbp = y_val[:, 1]

y_test_sbp = y_test[:, 0]
y_test_dbp = y_test[:, 1]

## RIDGE REGRESSION ##

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# Hyperparameter grid
param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
}

# =========================
# SBP
# =========================
grid_ridge_sbp = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=16,
    verbose=1
)

grid_ridge_sbp.fit(
    X_train_rocket,
    y_train[:, 0]
)

print("Best SBP parameters:")
print(grid_ridge_sbp.best_params_)

print("Best SBP CV RMSE:")
print((-grid_ridge_sbp.best_score_) ** 0.5)


# =========================
# DBP
# =========================
grid_ridge_dbp = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=16,
    verbose=1
)

grid_ridge_dbp.fit(
    X_train_rocket,
    y_train[:, 1]
)

print("\nBest DBP parameters:")
print(grid_ridge_dbp.best_params_)

print("Best DBP CV RMSE:")
print((-grid_ridge_dbp.best_score_) ** 0.5)

In [ ]:
y_pred_sbp = grid_ridge_sbp.predict(X_test_rocket)
y_pred_dbp = grid_ridge_dbp.predict(X_test_rocket)

In [ ]:
print("MiniRocket + Ridge — SBP")
print("------------------------")
print_metrics(y_test_sbp, y_pred_sbp)

print("\nMiniRocket + Ridge — DBP")
print("------------------------")
print_metrics(y_test_dbp, y_pred_dbp)

In [ ]:
plot_regression_results(
    y_test_sbp,
    y_pred_sbp,
    "MiniRocket + Ridge",
    "SBP"
)

plot_regression_results(
    y_test_dbp,
    y_pred_dbp,
    "MiniRocket + Ridge",
    "DBP"
)

In [ ]:
results_vitaldb = evaluate_minirocket_vitaldb(
    base_dir=BASE_DIR,
    rocket=rocket,
    best_en_sbp=grid_ridge_sbp,
    best_en_dbp=grid_ridge_dbp,
    batch_size=5000
)

## ELASTIC NET ##

In [ ]:
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

# --------------------------------------------------
# Use a subset ONLY for hyperparameter tuning due  to large dataset size
# --------------------------------------------------
rng = np.random.RandomState(42)

n_tune = min(20000, len(X_train_rocket))
tune_idx = rng.choice(
    len(X_train_rocket),
    size=n_tune,
    replace=False
)

X_tune = X_train_rocket.iloc[tune_idx] if hasattr(X_train_rocket, "iloc") else X_train_rocket[tune_idx]

y_tune_sbp = y_train[tune_idx, 0]
y_tune_dbp = y_train[tune_idx, 1]

print("ElasticNet tuning data:", X_tune.shape)

elastic_net = ElasticNet(
    max_iter=2000,
    random_state=42
)

param_grid_en = {
    "alpha": [0.01, 0.05, 0.1],
    "l1_ratio": [0.1, 0.5, 0.9]
}

grid_en_sbp = GridSearchCV(
    estimator=elastic_net,
    param_grid=param_grid_en,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=16,
    verbose=1
)

grid_en_sbp.fit(X_tune, y_tune_sbp)

print("Best SBP parameters:", grid_en_sbp.best_params_)
print(
    "Best SBP CV RMSE:",
    np.sqrt(-grid_en_sbp.best_score_)
)


grid_en_dbp = GridSearchCV(
    estimator=elastic_net,
    param_grid=param_grid_en,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=16,
    verbose=1
)

grid_en_dbp.fit(X_tune, y_tune_dbp)

print("Best DBP parameters:", grid_en_dbp.best_params_)
print(
    "Best DBP CV RMSE:",
    np.sqrt(-grid_en_dbp.best_score_)
)

In [ ]:
best_en_sbp = ElasticNet(
    alpha=grid_en_sbp.best_params_["alpha"],
    l1_ratio=grid_en_sbp.best_params_["l1_ratio"],
    max_iter=2000,
    random_state=42
)

best_en_dbp = ElasticNet(
    alpha=grid_en_dbp.best_params_["alpha"],
    l1_ratio=grid_en_dbp.best_params_["l1_ratio"],
    max_iter=2000,
    random_state=42
)

best_en_sbp.fit(X_train_rocket, y_train[:, 0])
best_en_dbp.fit(X_train_rocket, y_train[:, 1])

In [ ]:
print("SBP iterations:", best_en_sbp.n_iter_)
print("DBP iterations:", best_en_dbp.n_iter_)

In [ ]:
y_pred_en_sbp = best_en_sbp.predict(X_test_rocket)
y_pred_en_dbp = best_en_dbp.predict(X_test_rocket)

In [ ]:
print("ElasticNet — SBP")
print("----------------")
print_metrics(y_test[:, 0], y_pred_en_sbp)

print("\nElasticNet — DBP")
print("----------------")
print_metrics(y_test[:, 1], y_pred_en_dbp)

In [ ]:
plot_regression_results(
    y_test_sbp,
    y_pred_en_sbp,
    "MiniRocket + ElasticNet",
    "SBP"
)

plot_regression_results(
    y_test_dbp,
    y_pred_en_dbp,
    "MiniRocket + ElasticNet",
    "DBP"
)

In [ ]:
results_vitaldb = evaluate_minirocket_vitaldb(
    base_dir=BASE_DIR,
    rocket=rocket,
    best_en_sbp=best_en_sbp,
    best_en_dbp=best_en_dbp,
    batch_size=5000
)